# Multiview 2D+3D glaucoma experiment — OCT "report-like" projections + fusion

Idea: from a raw 200^3 optic-disc OCT volume we *project* a few 2D views that mimic the
panels of a standard OCT disc report (en-face reflectance, retinal-band slab views), then
train a **multi-branch net**: N standard-2D-CNN branches (one per view) + the original 3D
branch. Fusion of branch embeddings is ablated: **add / element-wise mul / concat /
self-attention**, plus single-branch baselines (3D-only and each 2D view alone).

Run on Colab GPU for the real experiment. A fast **SMOKE** path (synthetic volumes, tiny
models, CPU) validates every code path before spending GPU hours.


In [ ]:
import os, sys, json, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SMOKE = os.environ.get("MV_SMOKE", "0") == "1"
IN_COLAB = False
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except Exception:
    pass
if os.path.exists(".env"):
    for _line in open(".env"):
        _line = _line.strip()
        if _line and "=" in _line and not _line.startswith("#"):
            _k, _v = _line.split("=", 1)
            os.environ.setdefault(_k, _v)
if SMOKE:
    os.environ.setdefault("WANDB_MODE", "offline")
    os.environ.setdefault("WANDB_API_KEY", "local")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
AMP_DTYPE = torch.bfloat16 if (DEVICE.type == "cuda" and torch.cuda.is_bf16_supported()) else torch.float16
SEED = 42
print("smoke=", SMOKE, "device=", DEVICE, "amp=", USE_AMP)


In [ ]:
def set_seed(s=SEED):
    np.random.seed(s)
    torch.manual_seed(s)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(s)

def init_wandb(run_name, config=None):
    if os.environ.get("WANDB_MODE") != "offline" and not os.environ.get("WANDB_API_KEY"):
        print("[wandb] WANDB_API_KEY missing; continuing without cloud logging")
        return None
    try:
        import wandb
        return wandb.init(project="glaucoma-thesis", name=run_name,
                          config=config or {}, id=run_name, resume="allow")
    except Exception as e:
        print("[wandb] init failed:", e)
        return None

def metrics(y_true, y_pred):
    y_true = np.asarray(y_true); y_pred = np.asarray(y_pred)
    acc = float((y_true == y_pred).mean())
    tp = int(((y_pred == 1) & (y_true == 1)).sum())
    fp = int(((y_pred == 1) & (y_true == 0)).sum())
    fn = int(((y_pred == 0) & (y_true == 1)).sum())
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"acc": acc, "precision": prec, "recall": rec, "f1": f1}


In [ ]:
CFG = {
    "num_classes": 2,
    "data_root": "/content/glaucoma_hf_200",
    "save_dir": "/content/drive/MyDrive/MasterBKDN/Thesis/multiview",
    "epochs": 15,
    "batch_size": 2,
    "grad_accum": 8,
    "lr": 2e-4,
    "wd": 1e-4,
    "patience": 8,
    "log_every": 40,
    "enc3d_features": (24, 48, 96, 192),
    "enc2d_features": (64, 128, 256, 512),
    "enc2d_blocks": (2, 2, 2, 2),
    "latent": 256,
    "views": ["aip_full", "slab_aip", "slab_mip"],
    "slab_half": 16,
    "workers": 0,
}
if SMOKE:
    CFG.update({
        "epochs": 1, "batch_size": 1, "grad_accum": 1, "log_every": 5,
        "enc3d_features": (4, 8, 16, 32), "enc2d_features": (8, 16, 32, 64),
        "enc2d_blocks": (1, 1, 1, 1), "latent": 32, "workers": 0,
        "smoke_sz": 48, "n_smoke_train": 8, "n_smoke_val": 4, "n_smoke_test": 2,
    })


In [ ]:
def to_depth_last(vol, dz):
    if dz == 2:
        return vol
    others = [i for i in range(3) if i != dz]
    return np.transpose(vol, tuple(others) + (dz,))

def depth_axis(vol):
    f = vol.astype(np.float32)
    stds = [float(f.mean(axis=tuple(i for i in range(3) if i != ax)).std()) for ax in range(3)]
    return int(np.argmax(stds))

def project_views(volc, methods, half=16):
    S = volc.shape[2]
    half = min(int(half), max(2, S // 8))
    prof = volc.mean(axis=(0, 1)).astype(np.float32)
    peak = int(prof.argmax())
    lo, hi = max(0, peak - half), min(S, peak + half + 1)
    out = []
    for m in methods:
        if m == "aip_full":
            v = volc.mean(axis=2)
        elif m == "slab_aip":
            v = volc[:, :, lo:hi].mean(axis=2)
        elif m == "slab_mip":
            v = volc[:, :, lo:hi].max(axis=2)
        else:
            raise ValueError(m)
        out.append(v.astype(np.float32))
    return np.stack(out, axis=0), (lo, hi)

_REAL_VIEWS = os.environ.get("MV_REAL_VIEWS", "")
if _REAL_VIEWS and os.path.exists(_REAL_VIEWS):
    _rv = np.load(_REAL_VIEWS)
    print("real volume", _rv.shape, "depth_axis=", depth_axis(_rv))
    _rvc = to_depth_last(_rv, depth_axis(_rv))
    _vv, _rr = project_views(_rvc, CFG["views"], CFG["slab_half"])
    print("views shape", _vv.shape, "range", float(_vv.min()), float(_vv.max()), "slab", _rr)


In [ ]:
def make_synth_volume(S, rng):
    vol = rng.integers(0, 15, size=(S, S, S), dtype=np.uint8)
    r0 = S // 4
    r1 = min(S - 1, r0 + max(4, S // 5))
    slab = vol[:, :, r0:r1].astype(np.int16) + 45
    vol[:, :, r0:r1] = np.clip(slab, 0, 255).astype(np.uint8)
    lab = int(rng.integers(0, 2))
    c = S // 2
    h = max(3, S // 8)
    if lab:
        vol[c - h:c + h, c - h:c + h, r0:r1] = np.clip(
            vol[c - h:c + h, c - h:c + h, r0:r1].astype(np.int16) + 90, 0, 255).astype(np.uint8)
    return vol, lab

class MVRealDS(Dataset):
    def __init__(self, data_root, split, methods, half):
        self.methods, self.half = methods, half
        self.labels = np.load(os.path.join(data_root, f"{split}_labels.npy"))
        vp = os.path.join(data_root, f"{split}_volumes.npy")
        self.volumes = np.load(vp, mmap_mode="r")
        first = np.ascontiguousarray(self.volumes[0][0])
        self.dz = depth_axis(first)
        print(split, "n=", len(self.labels), "depth_axis=", self.dz)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        vol = np.ascontiguousarray(self.volumes[i][0])
        vol = to_depth_last(vol, self.dz)
        views, _ = project_views(vol, self.methods, self.half)
        x = torch.from_numpy(vol[None].astype(np.float32) / 255.0)
        v = torch.from_numpy(views / 255.0)
        y = torch.tensor(int(self.labels[i]), dtype=torch.long)
        return {"x": x, "v": v, "labels": y}

class MVSynthDS(Dataset):
    def __init__(self, cfg, n):
        rng = np.random.default_rng(0)
        S = cfg["smoke_sz"]
        self.items = []
        for _ in range(n):
            vol, lab = make_synth_volume(S, rng)
            views, _ = project_views(vol, cfg["views"], cfg["slab_half"])
            x = torch.from_numpy(vol[None].astype(np.float32) / 255.0)
            v = torch.from_numpy(views / 255.0)
            y = torch.tensor(lab, dtype=torch.long)
            self.items.append({"x": x, "v": v, "labels": y})

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        return self.items[i]

def make_loaders(cfg, split_counts=None):
    kw = dict(batch_size=cfg["batch_size"], num_workers=cfg["workers"],
              pin_memory=(DEVICE.type == "cuda"), drop_last=False)
    if cfg["workers"] > 0:
        kw.update(persistent_workers=True, prefetch_factor=4)
    if SMOKE:
        c = cfg if split_counts is None else split_counts
        tr = DataLoader(MVSynthDS(cfg, c["n_smoke_train"]), shuffle=True, **kw)
        va = DataLoader(MVSynthDS(cfg, c["n_smoke_val"]), shuffle=False, **kw)
        te = DataLoader(MVSynthDS(cfg, c["n_smoke_test"]), shuffle=False, **kw)
    else:
        dr = cfg["data_root"]
        tr = DataLoader(MVRealDS(dr, "Training", cfg["views"], cfg["slab_half"]), shuffle=True, **kw)
        va = DataLoader(MVRealDS(dr, "Validation", cfg["views"], cfg["slab_half"]), shuffle=False, **kw)
        te = DataLoader(MVRealDS(dr, "Test", cfg["views"], cfg["slab_half"]), shuffle=False, **kw)
    return tr, va, te


In [ ]:
def _gn(c, g=None):
    g = min(c, 8) if g is None else g
    while c % g != 0:
        g -= 1
    return nn.GroupNorm(g, c)

class ResBlock3D(nn.Module):
    def __init__(self, cin, cout, stride):
        super().__init__()
        self.c1 = nn.Conv3d(cin, cout, 3, stride=stride, padding=1)
        self.g1 = _gn(cout)
        self.c2 = nn.Conv3d(cout, cout, 3, padding=1)
        self.g2 = _gn(cout)
        self.short = (stride != (1, 1, 1) or cin != cout)
        if self.short:
            self.sc = nn.Conv3d(cin, cout, 1, stride=stride)
            self.sg = _gn(cout)
    def forward(self, x):
        r = F.relu(self.g1(self.c1(x)))
        r = self.g2(self.c2(r))
        if self.short:
            x = self.sg(self.sc(x))
        return F.relu(x + r)

class Enc3D(nn.Module):
    def __init__(self, features, depth_strides=(1, 1, 1, 2)):
        super().__init__()
        c0 = features[0]
        self.stem = nn.Sequential(nn.Conv3d(1, c0, 3, padding=1), _gn(c0), nn.ReLU())
        blocks = []
        for i, cout in enumerate(features):
            cin = features[i - 1] if i > 0 else c0
            lat = 1 if i == 0 else 2
            dep = depth_strides[i]
            blocks.append(ResBlock3D(cin, cout, (lat, lat, dep)))
        self.blocks = nn.Sequential(*blocks)
        self.out_dim = features[-1]
    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        return x.flatten(2).mean(2)

class ResBlock2D(nn.Module):
    def __init__(self, cin, cout, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(cin, cout, 3, stride=stride, padding=1, bias=False)
        self.g1 = _gn(cout)
        self.c2 = nn.Conv2d(cout, cout, 3, padding=1, bias=False)
        self.g2 = _gn(cout)
        self.short = (stride != 1 or cin != cout)
        if self.short:
            self.sc = nn.Conv2d(cin, cout, 1, stride=stride, bias=False)
            self.sg = _gn(cout)
    def forward(self, x):
        r = F.relu(self.g1(self.c1(x)))
        r = self.g2(self.c2(r))
        if self.short:
            x = self.sg(self.sc(x))
        return F.relu(x + r)

class Enc2D(nn.Module):
    def __init__(self, features, blocks):
        super().__init__()
        c0 = features[0]
        self.stem = nn.Sequential(nn.Conv2d(1, c0, 3, padding=1, bias=False), _gn(c0), nn.ReLU())
        layers = []
        for i, (cout, nb) in enumerate(zip(features, blocks)):
            cin = features[i - 1] if i > 0 else c0
            for b in range(nb):
                layers.append(ResBlock2D(cin if b == 0 else cout, cout, stride=(2 if (i > 0 and b == 0) else 1)))
        self.body = nn.Sequential(*layers)
        self.out_dim = features[-1]
    def forward(self, x):
        x = self.stem(x)
        x = self.body(x)
        return x.flatten(2).mean(2)

def _nheads(d):
    for h in (8, 4, 2, 1):
        if d % h == 0:
            return h
    return 1

class MultiBranch(nn.Module):
    def __init__(self, cfg, mode, view_index=None):
        super().__init__()
        self.mode = mode
        nv = len(cfg["views"])
        self.enc3d = Enc3D(cfg["enc3d_features"])
        self.enc2ds = nn.ModuleList([Enc2D(cfg["enc2d_features"], cfg["enc2d_blocks"]) for _ in range(nv)])
        d3 = self.enc3d.out_dim
        d2 = self.enc2ds[0].out_dim
        self.view_index = view_index
        if mode == "single3d":
            self.head = nn.Linear(d3, cfg["num_classes"])
            return
        if mode == "single2d":
            self.head = nn.Linear(d2, cfg["num_classes"])
            return
        D = cfg["latent"]
        dims = [d3] + [d2] * nv
        self.proj = nn.ModuleList([nn.Linear(d, D) for d in dims])
        if mode == "concat":
            self.head = nn.Linear(D * len(dims), cfg["num_classes"])
        elif mode == "add":
            self.head = nn.Linear(D, cfg["num_classes"])
        elif mode == "mul":
            self.head = nn.Linear(D, cfg["num_classes"])
        elif mode == "attn":
            layer = nn.TransformerEncoderLayer(d_model=D, nhead=_nheads(D),
                                              dim_feedforward=D * 2, dropout=0.1,
                                              batch_first=True, activation="relu")
            self.tf = nn.TransformerEncoder(layer, num_layers=1)
            self.cls = nn.Parameter(torch.randn(D))
            self.head = nn.Linear(D, cfg["num_classes"])
        else:
            raise ValueError(mode)
        self.D = D

    def embed(self, x, v):
        e3 = self.enc3d(x)
        e2 = [enc(v[:, i:i + 1]) for i, enc in enumerate(self.enc2ds)]
        return [e3] + e2

    def forward(self, x, v):
        if self.mode == "single3d":
            return self.head(self.enc3d(x))
        if self.mode == "single2d":
            return self.head(self.enc2ds[self.view_index](v[:, self.view_index:self.view_index + 1]))
        es = self.embed(x, v)
        if self.mode == "concat":
            f = torch.cat([F.relu(p(e)) for p, e in zip(self.proj, es)], dim=1)
            return self.head(f)
        toks = [F.relu(p(e)) for p, e in zip(self.proj, es)]
        if self.mode == "add":
            f = torch.stack(toks).sum(0)
            return self.head(f)
        if self.mode == "mul":
            f = torch.stack(toks).prod(0)
            return self.head(f)
        t = torch.stack(toks, dim=1)
        cls = self.cls.unsqueeze(0).unsqueeze(0).expand(t.shape[0], -1, -1)
        t = torch.cat([cls, t], dim=1)
        t = self.tf(t)
        return self.head(t[:, 0])

def build_model(cfg, spec):
    mode = spec["mode"]
    vi = spec.get("view")
    return MultiBranch(cfg, mode, vi)


In [ ]:
def _forward(model, batch):
    x = batch["x"].to(DEVICE)
    v = batch["v"].to(DEVICE)
    y = batch["labels"].to(DEVICE)
    if USE_AMP:
        with torch.autocast("cuda", dtype=AMP_DTYPE):
            logits = model(x, v)
    else:
        logits = model(x, v)
    return logits, y

def _eval(model, loader):
    model.eval()
    ys, ps, tot, n = [], [], 0.0, 0
    with torch.no_grad():
        for b in loader:
            logits, y = _forward(model, b)
            p = logits.argmax(1).cpu().numpy()
            ys.extend(y.cpu().numpy().tolist())
            ps.extend(p.tolist())
            tot += F.cross_entropy(logits, y).item() * y.numel()
            n += y.numel()
    return metrics(ys, ps), tot / max(n, 1)

def train_one(cfg, loaders, spec, wb=None, ckpt_dir=None):
    set_seed()
    name = spec["name"]
    tr, va, te = loaders
    model = build_model(cfg, spec).to(DEVICE)
    npar = sum(p.numel() for p in model.parameters()) / 1e6
    eff = cfg["batch_size"] * cfg["grad_accum"]
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"] * (eff / 4) ** 0.5, weight_decay=cfg["wd"])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=cfg["epochs"])
    scaler = torch.amp.GradScaler("cuda", enabled=(USE_AMP and AMP_DTYPE == torch.float16))
    crit = nn.CrossEntropyLoss()
    best_val, best_te, bad, step = 0.0, None, 0, 0
    t0 = time.time()
    print(f"[{name}] params={npar:.2f}M lr={opt.param_groups[0]['lr']:.2e}")
    for ep in range(cfg["epochs"]):
        model.train()
        opt.zero_grad(set_to_none=True)
        run_loss = 0.0
        for i, b in enumerate(tr):
            logits, y = _forward(model, b)
            loss = crit(logits, y) / cfg["grad_accum"]
            scaler.scale(loss).backward()
            run_loss += loss.item() * cfg["grad_accum"]
            step += 1
            if (i + 1) % cfg["grad_accum"] == 0:
                scaler.unscale_(opt)
                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(opt)
                scaler.update()
                opt.zero_grad(set_to_none=True)
            if wb is not None and step % max(1, cfg["log_every"]) == 0:
                wb.log({"train/loss": run_loss / (i + 1)}, step=step)
        sch.step()
        vm, vl = _eval(model, va)
        if wb is not None:
            wb.log({"epoch": ep + 1, "val/acc": vm["acc"], "val/loss": vl,
                    "val/f1": vm["f1"], "val/precision": vm["precision"],
                    "val/recall": vm["recall"], "val/best_acc": best_val}, step=step)
        imp = vm["acc"] > best_val
        print(f"[{name}] ep{ep+1:02d} train_loss={run_loss / max(len(tr),1):.4f} "
              + " ".join(f"val/{k}={v:.4f}" for k, v in vm.items()) + f" (best {best_val:.4f})")
        if imp:
            best_val, bad = vm["acc"], 0
            best_te, _ = _eval(model, te)
            if ckpt_dir:
                os.makedirs(ckpt_dir, exist_ok=True)
                torch.save(model.state_dict(), os.path.join(ckpt_dir, f"{name}.pt"))
        else:
            bad += 1
            if bad >= cfg["patience"]:
                break
    sec_ep = (time.time() - t0) / max(ep + 1, 1)
    res = {"name": name, "mode": spec["mode"], "view": spec.get("view"),
           "params": npar, "val": best_val, "test": (best_te or {}).get("acc", float("nan")),
           "test_metrics": best_te, "best_ep": int(ep + 1), "sec_ep": sec_ep, "status": "ok"}
    return res


In [ ]:
def load_rows(paths):
    for p in paths:
        try:
            with open(p) as fh:
                return json.load(fh)
        except Exception:
            continue
    return []

def save_rows(rows, paths):
    for p in paths:
        try:
            os.makedirs(os.path.dirname(p), exist_ok=True)
            with open(p, "w") as fh:
                json.dump(rows, fh, indent=2)
        except Exception as e:
            print("save skip", p, e)

if not SMOKE:
    CFG["workers"] = 2 if not (os.name == "nt") else 0
    RUN_SPECS = [
        {"mode": "single3d", "name": "single3d"},
        {"mode": "single2d", "view": 0, "name": "single2d-aip"},
        {"mode": "single2d", "view": 1, "name": "single2d-slabaip"},
        {"mode": "single2d", "view": 2, "name": "single2d-slabmip"},
        {"mode": "concat", "name": "fusion-concat"},
        {"mode": "add", "name": "fusion-add"},
        {"mode": "mul", "name": "fusion-mul"},
        {"mode": "attn", "name": "fusion-attn"},
    ]
    RUNS_LOCAL = "/content/multiview_results.json"
    RUNS_DRIVE = os.path.join(CFG["save_dir"], "multiview_results.json")
    tr, va, te = make_loaders(CFG)
    rows = load_rows([RUNS_LOCAL, RUNS_DRIVE])
    done = {r["name"] for r in rows}
    for sp in RUN_SPECS:
        if sp["name"] in done:
            print("skip (resume)", sp["name"])
            continue
        wb = init_wandb(sp["name"], config=dict(CFG, mode=sp["mode"], view=sp.get("view")))
        r = None
        try:
            r = train_one(CFG, (tr, va, te), sp, wb=wb, ckpt_dir=CFG["save_dir"])
        finally:
            if wb is not None and r is not None:
                wb.summary.update({"val": r.get("val"), "test": r.get("test")})
                wb.finish()
        rows.append(r)
        save_rows(rows, [RUNS_LOCAL, RUNS_DRIVE])
        print(f"  val={r['val']:.4f} test={r['test']:.4f} @ep{r['best_ep']}")
    ok = [r for r in rows if r.get("status") == "ok"]
    ok = sorted(ok, key=lambda r: -r["val"])
    print("\n===== MULTIVIEW 2D+3D (ranked by val) =====")
    for r in ok:
        print(f"{r['name']:16s} val={r['val']:.4f} test={r['test']:.4f} f1={r['test_metrics'].get('f1', float('nan')) if r['test_metrics'] else float('nan'):.4f}")
    if ok:
        print("\n>> WINNER:", ok[0]["name"])


In [ ]:
def run_smoke(cfg):
    print("===== SMOKE on CPU/small =====", flush=True)
    set_seed()
    cfg["workers"] = 0
    tr, va, te = make_loaders(cfg)
    b0 = next(iter(tr))
    print("batch keys", sorted(b0.keys()), "x", tuple(b0["x"].shape),
          "v", tuple(b0["v"].shape), "y", tuple(b0["labels"].shape))
    assert b0["v"].shape[1] == len(cfg["views"])
    specs = [{"mode": "single3d"}]
    for i in range(len(cfg["views"])):
        specs.append({"mode": "single2d", "view": i})
    for m in ("concat", "add", "mul", "attn"):
        specs.append({"mode": m})
    rows = []
    for sp in specs:
        sp = dict(sp)
        if sp["mode"] == "single2d":
            sp["name"] = f"s2d-{sp['view']}-{cfg['views'][sp['view']]}"
        else:
            sp["name"] = sp["mode"]
        wb = init_wandb("smoke-" + sp["name"], config=dict(cfg, **sp))
        r = train_one(cfg, (tr, va, te), sp, wb=wb)
        if wb is not None:
            wb.summary.update({k: v for k, v in r.items() if isinstance(v, (int, float))})
            wb.finish()
        rows.append(r)
        print(f"  RESULT {r['name']:12s} val={r['val']:.4f} test={r['test']:.4f}")
        assert r["val"] == r["val"] and r["test"] == r["test"], "NaN result"
    print("SMOKE_OK")

if SMOKE:
    run_smoke(CFG)


## How to run the real experiment (Colab GPU)
1. Run cells top to bottom. When `MV_SMOKE` is unset this runs the real experiment.
2. Set `CFG["data_root"]` to the folder with `Training|Validation|Test_volumes.npy`
   (raw 200^3) and optionally `CFG["save_dir"]` on Drive.
3. The last real cell sweeps baselines + 4 fusion ops (concat/add/mul/attn), resume-safe,
   each run logged to wandb project `glaucoma-thesis`.
4. To debug cheaply first: run `MV_SMOKE=1 python <this-as-script>` or a Colab cell with
   `os.environ['MV_SMOKE']='1'`.
